# 25 — Capstone 1: Sales Analytics Dashboard

End-to-end analytics project: raw data → cleaning → feature engineering → multi-dimensional
analysis → business insights. Covers every technique from Modules 1-5 in a single real project.

In [ ]:
import pandas as pd
import numpy as np
from io import StringIO

np.random.seed(42)

# ============================================================
# RAW DATA GENERATION — simulates messy data from 3 sources
# ============================================================

n = 2000

# Source 1: Orders
raw_orders = pd.DataFrame({
    'Order_ID':     range(10001, 10001 + n),
    'Customer_ID':  np.random.randint(1, 300, n),
    'Product_ID':   np.random.choice(['P001', 'P002', 'P003', 'P004', 'P005', 'P006'], n),
    'Quantity':     np.random.randint(1, 15, n),
    'Discount_Pct': np.random.choice([0, 5, 10, 15, 20, 25], n),
    'Region':       np.random.choice(['North', 'South', 'EAST', 'west', None], n,
                                      p=[0.3, 0.3, 0.2, 0.15, 0.05]),
    'Order_Date':   [f'{np.random.choice(["2022","2023"])}-{np.random.randint(1,13):02d}-{np.random.randint(1,29):02d}'
                     for _ in range(n)],
    'Ship_Days':    np.random.randint(1, 14, n)
})

# Introduce duplicates
raw_orders = pd.concat([raw_orders, raw_orders.sample(30, random_state=1)], ignore_index=True)

# Source 2: Products
products = pd.DataFrame({
    'Product_ID':    ['P001', 'P002', 'P003', 'P004', 'P005', 'P006'],
    'Product_Name':  ['Laptop Pro 15', 'iPhone 15', 'Galaxy Tab S9', 'AirPods Pro', 'USB-C Hub', 'Wireless Charger'],
    'Category':      ['Laptops', 'Smartphones', 'Tablets', 'Audio', 'Accessories', 'Accessories'],
    'Unit_Price':    [89999, 79999, 54999, 24999, 3999, 2499],
    'Cost_Price':    [60000, 55000, 38000, 15000, 2000, 1200],
})

# Source 3: Customers
customers = pd.DataFrame({
    'Customer_ID':    range(1, 301),
    'Customer_Name':  [f'Customer_{i:03d}' for i in range(1, 301)],
    'Segment':        np.random.choice(['Enterprise', 'SMB', 'Consumer'], 300, p=[0.2, 0.3, 0.5]),
    'City':           np.random.choice(['Mumbai', 'Delhi', 'Bangalore', 'Pune', 'Chennai',
                                         'Hyderabad', 'Kolkata', 'Ahmedabad'], 300),
})

print(f"raw_orders: {raw_orders.shape}")
print(f"products:   {products.shape}")
print(f"customers:  {customers.shape}")

## Phase 1: Data Quality Audit

In [ ]:
def audit_dataframe(df: pd.DataFrame, name: str = 'DataFrame') -> None:
    print(f"\n{'='*50}")
    print(f"AUDIT: {name}")
    print(f"Shape: {df.shape}")
    print(f"Dtypes:\n{df.dtypes.to_string()}")
    
    missing = df.isna().sum()
    if missing.any():
        print(f"\nMissing:\n{missing[missing > 0].to_string()}")
    
    dups = df.duplicated().sum()
    print(f"\nDuplicate rows: {dups}")
    print('='*50)

audit_dataframe(raw_orders, 'raw_orders')
audit_dataframe(products, 'products')
audit_dataframe(customers, 'customers')

## Phase 2: Cleaning Pipeline

In [ ]:
def clean_orders(df: pd.DataFrame) -> pd.DataFrame:
    return (
        df
        .drop_duplicates(subset=['Order_ID'])
        .assign(
            Region     = lambda d: d['Region'].str.title().fillna('Unknown'),
            Order_Date = lambda d: pd.to_datetime(d['Order_Date'], errors='coerce'),
        )
        .dropna(subset=['Order_Date'])
        .reset_index(drop=True)
    )

orders = clean_orders(raw_orders)
print(f"After cleaning: {orders.shape}")
print(orders.dtypes)
orders.head(3)

## Phase 3: Enrichment — Join All Sources

In [ ]:
enriched = (
    orders
    .merge(products, on='Product_ID', how='left', validate='m:1')
    .merge(customers, on='Customer_ID', how='left', validate='m:1')
)

print(f"Enriched shape: {enriched.shape}")
print(enriched.head(3))

## Phase 4: Feature Engineering

In [ ]:
final = (
    enriched
    .assign(
        # Revenue & profit
        gross_revenue  = lambda df: df['Unit_Price'] * df['Quantity'],
        discount_amt   = lambda df: df['gross_revenue'] * df['Discount_Pct'] / 100,
        net_revenue    = lambda df: df['gross_revenue'] - df['discount_amt'],
        cost           = lambda df: df['Cost_Price'] * df['Quantity'],
        gross_profit   = lambda df: df['net_revenue'] - df['cost'],
        gross_margin   = lambda df: (df['gross_profit'] / df['net_revenue'] * 100).round(1),

        # Date features
        year           = lambda df: df['Order_Date'].dt.year,
        month          = lambda df: df['Order_Date'].dt.month,
        quarter        = lambda df: df['Order_Date'].dt.quarter,
        day_of_week    = lambda df: df['Order_Date'].dt.day_name(),
        is_weekend     = lambda df: df['Order_Date'].dt.day_of_week >= 5,

        # Ship date
        ship_date      = lambda df: df['Order_Date'] + pd.to_timedelta(df['Ship_Days'], unit='D'),

        # Customer value tier
        value_tier     = lambda df: pd.cut(
                             df['net_revenue'],
                             bins=[0, 10000, 50000, 200000, 2000000],
                             labels=['Bronze', 'Silver', 'Gold', 'Platinum']
                         ),
    )
)

print(f"Final shape: {final.shape}")
print(final[['Order_ID', 'Product_Name', 'Quantity', 'gross_revenue', 'net_revenue',
              'gross_profit', 'gross_margin']].head(5))

## Phase 5: Business Analysis

In [ ]:
# ── Revenue & Profit by Category ──
by_category = (
    final
    .groupby('Category')
    .agg(
        orders        = ('Order_ID', 'count'),
        units_sold    = ('Quantity', 'sum'),
        gross_revenue = ('gross_revenue', 'sum'),
        net_revenue   = ('net_revenue', 'sum'),
        gross_profit  = ('gross_profit', 'sum'),
        avg_margin    = ('gross_margin', 'mean'),
        avg_discount  = ('Discount_Pct', 'mean')
    )
    .round(0)
    .sort_values('net_revenue', ascending=False)
)
print("Revenue & Profit by Category:")
print(by_category)

In [ ]:
# ── YoY Revenue ──
yoy = (
    final
    .groupby(['year', 'Category'])['net_revenue']
    .sum()
    .unstack('year')
    .round(0)
)
yoy.columns = [f'revenue_{y}' for y in yoy.columns]
yoy['yoy_growth_pct'] = ((yoy['revenue_2023'] - yoy['revenue_2022']) / yoy['revenue_2022'] * 100).round(1)
print("\nYoY Revenue by Category:")
print(yoy)

In [ ]:
# ── Regional Performance ──
by_region = (
    final
    .groupby('Region')
    .agg(
        orders=('Order_ID', 'count'),
        net_revenue=('net_revenue', 'sum'),
        avg_order_value=('net_revenue', 'mean')
    )
    .round(0)
    .sort_values('net_revenue', ascending=False)
)
# Add revenue share
by_region['revenue_share_pct'] = (by_region['net_revenue'] / by_region['net_revenue'].sum() * 100).round(1)
print("\nRegional Performance:")
print(by_region)

In [ ]:
# ── Top 10 Products ──
top10_products = (
    final
    .groupby('Product_Name')
    .agg(
        units=('Quantity', 'sum'),
        net_revenue=('net_revenue', 'sum'),
        gross_profit=('gross_profit', 'sum'),
        avg_margin=('gross_margin', 'mean')
    )
    .round(0)
    .sort_values('net_revenue', ascending=False)
)
print("\nTop Products by Revenue:")
print(top10_products)

In [ ]:
# ── Segment Analysis ──
segment_analysis = (
    final
    .groupby('Segment')
    .agg(
        customers=('Customer_ID', 'nunique'),
        orders=('Order_ID', 'count'),
        net_revenue=('net_revenue', 'sum'),
        avg_order_size=('net_revenue', 'mean'),
        avg_discount=('Discount_Pct', 'mean')
    )
    .round(0)
)
segment_analysis['orders_per_customer'] = (segment_analysis['orders'] / segment_analysis['customers']).round(1)
print("\nCustomer Segment Analysis:")
print(segment_analysis)

## Phase 6: Time Series Analysis

In [ ]:
# Monthly revenue trend with rolling average
ts = (
    final
    .set_index('Order_Date')
    .sort_index()
    ['net_revenue']
    .resample('ME')
    .sum()
    .to_frame()
)
ts['rolling_3m'] = ts['net_revenue'].rolling(3, min_periods=1).mean().round(0)
ts['mom_growth_pct'] = ts['net_revenue'].pct_change(1).round(3) * 100

print("Monthly Revenue:")
print(ts.round(0))

## Phase 7: Pivot Dashboard

In [ ]:
# Category × Region revenue matrix
dashboard = pd.pivot_table(
    final,
    values=['net_revenue', 'gross_profit'],
    index='Category',
    columns='Region',
    aggfunc='sum',
    margins=True,
    margins_name='Total'
).round(0)

print("Category × Region Dashboard:")
print(dashboard['net_revenue'])

In [ ]:
# Quarterly quarter-by-category matrix
q_matrix = pd.pivot_table(
    final[final['year'] == 2023],
    values='net_revenue',
    index='Category',
    columns='quarter',
    aggfunc='sum',
    fill_value=0
).round(0)
q_matrix.columns = [f'Q{c}' for c in q_matrix.columns]
q_matrix['Total_2023'] = q_matrix.sum(axis=1)
print("2023 Quarterly Revenue by Category:")
print(q_matrix)

## Phase 8: Customer RFM Analysis

In [ ]:
# RFM — Recency, Frequency, Monetary
snapshot_date = final['Order_Date'].max() + pd.Timedelta('1D')

rfm = (
    final
    .groupby('Customer_ID')
    .agg(
        recency   = ('Order_Date', lambda x: (snapshot_date - x.max()).days),
        frequency = ('Order_ID', 'count'),
        monetary  = ('net_revenue', 'sum')
    )
    .round(0)
)

# Score each dimension (5=best, 1=worst)
rfm['R_score'] = pd.qcut(rfm['recency'],   q=5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['M_score'] = pd.qcut(rfm['monetary'],  q=5, labels=[1, 2, 3, 4, 5]).astype(int)

rfm['RFM_score'] = rfm['R_score'] + rfm['F_score'] + rfm['M_score']

rfm['customer_segment'] = pd.cut(
    rfm['RFM_score'],
    bins=[2, 7, 10, 13, 16],
    labels=['At Risk', 'Occasional', 'Loyal', 'Champion']
)

print("RFM Segment Distribution:")
print(rfm['customer_segment'].value_counts())
print("\nRFM Summary by Segment:")
print(rfm.groupby('customer_segment', observed=True)[['recency', 'frequency', 'monetary']].mean().round(0))

## Phase 9: Executive Summary

In [ ]:
print("="*60)
print("EXECUTIVE SUMMARY")
print("="*60)
print(f"  Total Orders:         {len(final):,}")
print(f"  Unique Customers:     {final['Customer_ID'].nunique():,}")
print(f"  Unique Products:      {final['Product_ID'].nunique()}")
print(f"  Total Gross Revenue:  ₹{final['gross_revenue'].sum():,.0f}")
print(f"  Total Net Revenue:    ₹{final['net_revenue'].sum():,.0f}")
print(f"  Total Gross Profit:   ₹{final['gross_profit'].sum():,.0f}")
print(f"  Avg Gross Margin:     {final['gross_margin'].mean():.1f}%")
print(f"  Avg Order Value:      ₹{final['net_revenue'].mean():,.0f}")
print(f"  Avg Discount Given:   {final['Discount_Pct'].mean():.1f}%")
print(f"  Avg Ship Time:        {final['Ship_Days'].mean():.1f} days")
print()
print(f"  Top Category:         {by_category['net_revenue'].idxmax()}")
print(f"  Top Region:           {by_region['net_revenue'].idxmax()}")
print(f"  Top Segment:          {segment_analysis['net_revenue'].idxmax()}")
print("="*60)